John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

In [ ]:
library(tidyverse)

: 

In [ ]:
rawdata <- read_csv("data/megazord.csv")

New names:
• `` -> `...1`
Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 12799 Columns: 127
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (14): ID, recordist, Begin.Path, Location_Number, Comments, ST1, Data_...
dbl  (104): ...1, X.2, X.1, selec, start, end, File.Offset..s., SNR.NIST.Qui...
lgl    (2): RepeatST, GreaterThanT
dttm   (1): DateTime
date   (1): date
time   (5): click_clock, start_hms, click_clock_hms, clock_time, sunrise

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [ ]:
columns_of_interest <- c(
    "ID", # Which male is singing (each has a unique identifier)
    "DateTime", # Date AND Time
    "date",
    "time_rel_sunrise", # How much time has passed since sunrise
    "UST", # Universal Song Type
    "IST", # Individual Song Type 
    "D_M1_M2", "D_M1_M3", "D_M1_M4", "D_M1_M5", "D_M1_M6", "D_M2_M3", "D_M2_M4", "D_M2_M5", "D_M2_M6", "D_M3_M4", "D_M3_M5", "D_M3_M6", "D_M4_M5", "D_M4_M6", "D_M5_M6", # IID - inter-individual distance between Males 
    "M1_Lat", "M1_Lon", "M2_Lat", "M2_Lon", "M3_Lat", "M3_Lon", "M4_Lat", "M4_Lon", "M5_Lat", "M5_Lon", "M6_Lat", "M6_Lon", # Latitude and Longitude
    "M1_last_UST", "M2_last_UST", "M3_last_UST", "M4_last_UST", "M5_last_UST", "M6_last_UST", # Last song type
    "M1_match", "M2_match", "M3_match", "M4_match", "M5_match", "M6_match",  # Does focal male match with Male X's last song? (Cannot match with itself)
    "number_of_matches", # How many males' last song matches with the focal male's song?
    "does_it_match", # Does the focal male's current song match with any other male's most recent song?
    "neighbourhood", # North, Centre, or South. We'll be starting our analysis on the North neighbourhood
    "Location_Number", 
    "Time_Diff", # Time since this bird's own last song
    "RepeatST", # Is this a repeat of a song the bird just sang?
    "Repeat_ST_Run" # How many songs of the same type have been repeated in this run? ("Streak")
    # "Phase" # DC (Dawn Chorus) or PDC (Post-Dawn Chorus) TODO: Filter out PDC data
    # "Data_Class" # We are only interested in "song"
    )

In [ ]:
data <- rawdata %>% 
    filter(Data_Class == "song") %>% # Only interested in song data
    filter(Phase == "DC") %>% # Only interested in Dawn Chorus songs
    # filter(LOKI == 0, acoustic_outlier == 0) %>% # Remove Loggerhead Kingbird interference and acoustic outliers # NOTE: only do this if doing acoustic analysis
    select(columns_of_interest) %>% 
    arrange(desc("DateTime")) %>%
    mutate(across(contains("UST"), ~ as.factor(.x))) # Song type is categorical attribute, not numerical

summary(data)

      ID               DateTime                        date           
 Length:11694       Min.   :2022-05-09 05:19:33   Min.   :2022-05-09  
 Class :character   1st Qu.:2022-05-14 05:17:16   1st Qu.:2022-05-14  
 Mode  :character   Median :2022-05-17 05:40:17   Median :2022-05-17  
                    Mean   :2022-05-17 02:51:34   Mean   :2022-05-16  
                    3rd Qu.:2022-05-20 05:33:49   3rd Qu.:2022-05-20  
                    Max.   :2022-05-24 06:04:02   Max.   :2022-05-24  
                                                                      
 time_rel_sunrise       UST           IST               D_M1_M2      
 Min.   :-2840.9   1      : 785   Length:11694       Min.   :  0.00  
 1st Qu.:-1675.8   10     : 768   Class :character   1st Qu.: 58.74  
 Median :-1301.0   9      : 747   Mode  :character   Median : 94.04  
 Mean   :-1258.6   8      : 667                      Mean   : 92.45  
 3rd Qu.: -902.4   13     : 627                      3rd Qu.: 94.08  
 Max.   : 11

In [ ]:
# # Add a feature for focal male's M number
# north_ids = list("1" = "LbOW", "2" = "RbDgY", "3" = "YKY", "4" = "TbTbLg", "5" = "DgTbO", "6" = "RRO")
# centre_ids <- list("1" = "TbWY", "2" = "KRDg", "3" = "OPR", "4" = "WLpTb", "5" = "KLgK", "6" = "DgTbO")
# # TODO: 
# data <- data %>% mutate(
#      mutate("{n_last_song}" := if_else(ID == ids[[i]], lag(UST, n-1), NA)) %>%
#     male_number <- if_else(neighbourhood == "north", )
# )

In [ ]:
# data <- data %>% 
#     mutate(male_number = case_match(neighbourhood,
#         as.factor("north") ~ case_match(ID,
#             "1" ~ "LbOW", 
#             "2" ~ "RbDgY", 
#             "3" ~ "YKY", 
#             "4" ~ "TbTbLg", 
#             "5" ~ "DgTbO", 
#             "6" ~ "RRO"
#         ),
#         as.factor("centre") ~ case_match(ID,
#             "1" ~ "TbWY", 
#             "2" ~ "KRDg", 
#             "3" ~ "OPR", 
#             "4" ~ "WLpTb", 
#             "5" ~ "KLgK", 
#             "6" ~ "DgTbO"
#         )
#     ))

In [ ]:
# TODO: Figure this out
lookup_table <- tibble::tribble(
    ~neighbourhood, ~ID, ~male_number,
    "north",  "1", "LbOW",
    "north",  "2", "RbDgY",
    "north",  "3", "YKY",
    "north",  "4", "TbTbLg",
    "north",  "5", "DgTbO",
    "north",  "6", "RRO",
    "centre", "1", "TbWY",
    "centre", "2", "KRDg",
    "centre", "3", "OPR",
    "centre", "4", "WLpTb",
    "centre", "5", "KLgK",
    "centre", "6", "DgTbO"
)
data <- data %>%
  left_join(lookup_table, by = c("neighbourhood", "ID"))

In [ ]:
data

ID,DateTime,date,time_rel_sunrise,UST,IST,D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,⋯,M5_match,M6_match,number_of_matches,does_it_match,neighbourhood,Location_Number,Time_Diff,RepeatST,Repeat_ST_Run,male_number
<chr>,<dttm>,<date>,<dbl>,<fct>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<lgl>,<dbl>,<chr>
DgTbO,2022-05-13 05:24:14,2022-05-13,-1935.691,25,DgTbO_13,NA,NA,NA,NA,⋯,0,0,0,0,north,42,0,FALSE,1,NA
DgTbO,2022-05-13 05:24:40,2022-05-13,-1909.821,20,DgTbO_1,NA,NA,NA,NA,⋯,0,0,0,0,north,42,26,FALSE,1,NA
DgTbO,2022-05-13 05:24:48,2022-05-13,-1901.666,1,DgTbO_6,NA,NA,NA,NA,⋯,0,0,0,0,north,42,8,FALSE,1,NA
DgTbO,2022-05-13 05:24:59,2022-05-13,-1890.748,153,DgTbO_4,NA,NA,NA,NA,⋯,0,0,0,0,north,42,11,FALSE,1,NA
DgTbO,2022-05-13 05:25:09,2022-05-13,-1880.963,33,DgTbO_5,NA,NA,NA,NA,⋯,0,0,0,0,north,42,10,FALSE,1,NA
DgTbO,2022-05-13 05:25:18,2022-05-13,-1871.335,8,DgTbO_7,NA,NA,NA,NA,⋯,0,1,1,1,north,42,9,FALSE,1,NA
DgTbO,2022-05-13 05:25:26,2022-05-13,-1863.036,33,DgTbO_5,NA,NA,NA,NA,⋯,0,0,0,0,north,42,8,FALSE,1,NA
DgTbO,2022-05-13 05:25:35,2022-05-13,-1854.549,20,DgTbO_1,NA,NA,NA,NA,⋯,0,0,0,0,north,42,9,FALSE,1,NA
DgTbO,2022-05-13 05:25:41,2022-05-13,-1848.341,7,DgTbO_2,NA,NA,NA,NA,⋯,0,0,0,0,north,42,6,FALSE,1,NA


### Export Data

In [ ]:
write_csv(data, "../data/minizord.csv")

ERROR: Error: Cannot open file for writing:
* 'c:/Users/johna/OneDrive/Documents/data/minizord.csv'


In [ ]:
# This cell used for easy copy/paste of column names

"...1"
"X.2"
"X.1"
"selec"
"start"
"end"
"date"
"ID"
"recordist"
"Begin.Path"
"File.Offset..s."
"SNR.NIST.Quick..dB."
"Location_Number"
"Comments"
"ST1"
"Data_Class"
"selec.file"
"sound.files"
"click_clock"
"click_rec"
"start_hms"
"click_clock_hms"
"click_rec_hms"
"clock_time"
"sunrise"
"time_rel_sunrise"
"M1_Off"
"M2_Off"
"M3_Off"
"M4_Off"
"M5_Off"
"M6_Off"
"Lat"
"Lon"
"M1_Lat"
"M1_Lon"
"M2_Lat"
"M2_Lon"
"M3_Lat"
"M3_Lon"
"M4_Lat"
"M4_Lon"
"M5_Lat"
"M5_Lon"
"M6_Lat"
"M6_Lon"
"D_M1_M2"
"D_M1_M3"
"D_M1_M4"
"D_M1_M5"
"D_M1_M6"
"D_M2_M3"
"D_M2_M4"
"D_M2_M5"
"D_M2_M6"
"D_M3_M4"
"D_M3_M5"
"D_M3_M6"
"D_M4_M5"
"D_M4_M6"
"D_M5_M6"
"IST"
"X"
"UST"
"M1_last_UST"
"M2_last_UST"
"M3_last_UST"
"M4_last_UST"
"M1_match"
"M2_match"
"M3_match"
"M4_match"
"neighbourhood"
"M5_last_UST"
"M6_last_UST"
"M5_match"
"M6_match"
"number_of_matches"
"does_it_match"
"song_end_time"
"DateTime"
"Bird_Date"
"Time_Diff"
"RepeatST"
"GreaterThanT"
"Repeat_ST_Run"
"Long_Latency_Run"
"meets_lat_thresh"
"meets_rep_thresh"
"meets_either_thresh"
"Phase"
"End_DC"
"index"
"orig_start"
"orig_end"
"rms"
"LOKI"
"note_count"
"song_start"
"song_end"
"song_duration"
"note_duration"
"gap_duration"
"percent_sound"
"min_freq"
"max_freq"
"peak_freq"
"max_note_dur"
"mean_freq"
"last_note_dur"
"freq_BW_subtraction"
"freq_BW_ratio"
"TR_unbiased"
"mean_note_dur"
"mean_gap_dur"
"acoustic_outlier"
"avg_note_dur_nl"
"avg_gap_dur_nl"
"avg_note_gap_ratio_nl"
"avg_note_gap_difference_nl"
"avg_min_freq_nl"
"avg_max_freq_nl"
"avg_note_bw_ratio_nl"
"avg_note_FM_nl"
"avg_silent_jump_ratio_nl"
"avg_silent_gap_FM_nl"
"avg_recovery_time_nl"


[1] "...1"

[1] "X.2"

[1] "X.1"

[1] "selec"

[1] "start"

[1] "end"

... (117 outputs left out)

[1] "avg_note_FM_nl"

[1] "avg_silent_jump_ratio_nl"

[1] "avg_silent_gap_FM_nl"

[1] "avg_recovery_time_nl"